# Variant Calling and SNP Analysis

## Tier 3 - Applied Bioinformatics

This notebook covers the complete variant analysis pipeline: from aligned reads to annotated variants, including VCF format, variant filtering, annotation databases, population genetics, and GWAS concepts.

### Learning Objectives
- Classify variant types: SNPs, indels, structural variants, CNVs
- Understand the variant calling pipeline
- Parse and work with VCF format in depth
- Apply variant quality filters
- Annotate variants using databases (ClinVar, dbSNP, gnomAD)
- Understand population genetics concepts (allele frequency, HWE)
- Interpret GWAS results: Manhattan plots, QQ plots

---

---

[< Previous: Bioinformatics Data Formats: A Comprehensive Guide](../01_NGS_Fundamentals/02_bio_data_formats.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: SNP Calling Pipeline: A Practical Hands-On Work... >](02_snp_calling_pipeline.ipynb)

---

## 1. Types of Genetic Variants

### 1.1 Small Variants

| Type | Description | Size | Example |
|------|------------|------|---------|
| **SNP/SNV** | Single nucleotide change | 1 bp | A -> G |
| **MNP/MNV** | Multiple adjacent nucleotide changes | 2+ bp | AT -> GC |
| **Insertion** | Bases added | 1-50 bp | A -> ATCG |
| **Deletion** | Bases removed | 1-50 bp | ATCG -> A |
| **Indel** | Insertion or deletion | 1-50 bp | Complex |

### 1.2 Structural Variants (SVs)

| Type | Description | Size | Detection |
|------|------------|------|-----------|
| **Large deletion** | Loss of genomic segment | >50 bp | Split reads, depth changes |
| **Large insertion** | Gain of genomic segment | >50 bp | Split reads |
| **Duplication** | Segment copied | >50 bp | Increased depth |
| **Inversion** | Segment reversed | >50 bp | Discordant read pairs |
| **Translocation** | Segment moved between chromosomes | Variable | Discordant read pairs |

### 1.3 Copy Number Variants (CNVs)

```
Normal:       [====A====][====B====][====C====]
Deletion:     [====A====]           [====C====]   (loss of B)
Duplication:  [====A====][====B====][====B====][====C====]  (gain of B)
Amplification:[====A====][=B=][=B=][=B=][=B=] [====C====]  (multiple copies)

CNV detection: read depth analysis
  Normal region:  |||||||||||||||  (expected depth)
  Deletion:       ||| ||| ||       (reduced depth)
  Duplication:    |||||||||||||||||||||||  (increased depth)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re
import math
import random

random.seed(42)
np.random.seed(42)

def classify_variant(ref, alt):
    """Classify a variant based on REF and ALT alleles."""
    if len(ref) == 1 and len(alt) == 1:
        return 'SNV'
    elif len(ref) > 1 and len(alt) > 1 and len(ref) == len(alt):
        return 'MNV'
    elif len(ref) < len(alt):
        return 'insertion'
    elif len(ref) > len(alt):
        return 'deletion'
    else:
        return 'complex'

# Examples
variants = [
    ('A', 'G', 'rs334'),
    ('AT', 'GC', 'rs12345'),
    ('A', 'ATCG', '.'),
    ('ATCG', 'A', '.'),
    ('C', 'T', 'rs7903146'),
]

print(f"{'REF':>6} {'ALT':>6} {'Type':>12} {'ID':>12}")
print("-" * 42)
for ref, alt, vid in variants:
    vtype = classify_variant(ref, alt)
    print(f"{ref:>6} {alt:>6} {vtype:>12} {vid:>12}")

---

## 2. Variant Calling Pipeline

### 2.1 Pipeline Overview

```
Aligned BAM file
       |
       v
  Mark duplicates (Picard / samtools markdup)
       |
       v
  Base quality score recalibration (GATK BQSR) [optional]
       |
       v
  Variant calling (GATK HaplotypeCaller / bcftools / FreeBayes)
       |
       v
  Raw VCF
       |
       v
  Variant filtering (GATK VQSR / hard filters)
       |
       v
  Filtered VCF
       |
       v
  Annotation (VEP / ANNOVAR / SnpEff)
       |
       v
  Annotated variants for interpretation
```

### 2.2 Variant Calling Tools

| Tool | Algorithm | Best For | Notes |
|------|-----------|----------|-------|
| **GATK HaplotypeCaller** | Local de novo assembly | WGS, WES | Gold standard, part of GATK Best Practices |
| **bcftools mpileup/call** | Pileup-based | WGS, quick analysis | Fast, lightweight |
| **FreeBayes** | Bayesian haplotype-based | WGS, small cohorts | Good for low-frequency variants |
| **DeepVariant** | Deep learning | WGS, WES | Google's neural network caller |
| **Strelka2** | Statistical model | Somatic variants | Good for tumor-normal pairs |

### 2.3 How GATK HaplotypeCaller Works

1. **Identify active regions**: Find regions where reads suggest variants
2. **Local reassembly**: Build a de Bruijn graph of reads in the region
3. **Haplotype determination**: Extract candidate haplotypes from the graph
4. **Pairwise alignment**: Align reads to each candidate haplotype
5. **Genotype assignment**: Use Bayesian model to assign genotypes

```bash
# GATK Best Practices pipeline
# NOTE: GATK requires read groups in the BAM file.
# When aligning with BWA, always include the -R flag:
#   bwa mem -R '@RG\tID:sample1\tSM:sample1\tPL:ILLUMINA\tLB:lib1' \
#       ref.fa R1.fq.gz R2.fq.gz | samtools sort -o sorted.bam
# Without read groups, MarkDuplicates and HaplotypeCaller will fail.

# 1. Mark duplicates
gatk MarkDuplicates -I sorted.bam -O dedup.bam -M metrics.txt

# 2. Base quality score recalibration
gatk BaseRecalibrator -I dedup.bam -R ref.fa --known-sites dbsnp.vcf -O recal.table
gatk ApplyBQSR -I dedup.bam -R ref.fa --bqsr-recal-file recal.table -O recal.bam

# 3. Call variants
gatk HaplotypeCaller -I recal.bam -R ref.fa -O raw.g.vcf -ERC GVCF

# 4. Joint genotyping (for cohorts)
gatk GenotypeGVCFs -R ref.fa -V raw.g.vcf -O genotyped.vcf

# 5. Filter variants
gatk VariantFiltration -R ref.fa -V genotyped.vcf \
  --filter-expression "QD < 2.0" --filter-name "LowQD" \
  --filter-expression "FS > 60.0" --filter-name "StrandBias" \
  --filter-expression "MQ < 40.0" --filter-name "LowMQ" \
  -O filtered.vcf
```

---

## 3. VCF Format in Depth

### 3.1 VCF Structure

```
## Meta-information lines (start with ##)
##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Read Depth">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Allele Count">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Total Allele Number">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Sample Read Depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic Depths">
##FILTER=<ID=LowQual,Description="Low quality">

## Header line (starts with #)
#CHROM  POS  ID  REF  ALT  QUAL  FILTER  INFO  FORMAT  SAMPLE1  SAMPLE2

## Data lines
chr1  10000  rs123  A  G  5000  PASS  DP=200;AF=0.50;AC=1;AN=2  GT:DP:GQ:AD  0/1:100:99:50,50
```

### 3.2 Column Details

| Column | Description | Notes |
|--------|-------------|-------|
| CHROM | Chromosome | chr1, chr2, ... |
| POS | 1-based position | Position of REF allele |
| ID | Variant ID | rsID or . if novel |
| REF | Reference allele | Must match reference genome |
| ALT | Alternate allele(s) | Comma-separated for multi-allelic |
| QUAL | Phred-scaled quality | -10*log10(P[no variant]) |
| FILTER | Filter status | PASS or semicolon-separated filters |
| INFO | Variant-level annotations | Key=Value pairs, semicolon-separated |
| FORMAT | Sample field format | Colon-separated field names |
| SAMPLE | Sample genotype data | Colon-separated values matching FORMAT |

### 3.3 Genotype Encodings

| Genotype | Meaning | Zygosity |
|----------|---------|----------|
| 0/0 | Homozygous reference | - |
| 0/1 | Heterozygous | Het |
| 1/1 | Homozygous alternate | Hom-alt |
| 1/2 | Heterozygous for two different alts | Het (multi-allelic) |
| ./. | Missing / no call | - |
| 0|1 | Phased heterozygous | Het (phased) |

In [ ]:
def parse_vcf(vcf_text):
    """
    Parse VCF text into structured data.
    
    Returns:
        meta: list of meta-information lines
        header: list of column names
        variants: list of variant dictionaries
    """
    meta = []
    header = []
    variants = []
    
    for line in vcf_text.strip().split('\n'):
        if line.startswith('##'):
            meta.append(line)
        elif line.startswith('#'):
            header = line[1:].split('\t')
        else:
            fields = line.split('\t')
            if len(fields) < 8:
                continue
            
            # Parse INFO field
            info = {}
            for item in fields[7].split(';'):
                if '=' in item:
                    key, value = item.split('=', 1)
                    info[key] = value
                else:
                    info[item] = True  # Flag fields
            
            variant = {
                'chrom': fields[0],
                'pos': int(fields[1]),
                'id': fields[2],
                'ref': fields[3],
                'alt': fields[4].split(','),  # Multiple alts possible
                'qual': float(fields[5]) if fields[5] != '.' else None,
                'filter': fields[6],
                'info': info,
                'type': classify_variant(fields[3], fields[4].split(',')[0]),
            }
            
            # Parse sample genotypes if present
            if len(fields) > 9:
                fmt_fields = fields[8].split(':')
                samples = {}
                for i, sample_data in enumerate(fields[9:]):
                    sample_values = sample_data.split(':')
                    sample_dict = {}
                    for j, key in enumerate(fmt_fields):
                        if j < len(sample_values):
                            sample_dict[key] = sample_values[j]
                    samples[f'SAMPLE{i+1}'] = sample_dict
                variant['samples'] = samples
            
            variants.append(variant)
    
    return meta, header, variants


# Example VCF with multiple samples and rich annotations
vcf_example = """##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Allele Count">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Total Alleles">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic Depths">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSAMPLE1\tSAMPLE2\tSAMPLE3
chr1\t10000\trs123456\tA\tG\t5000\tPASS\tDP=300;AF=0.33;AC=2;AN=6\tGT:DP:GQ:AD\t0/1:100:99:50,50\t0/0:95:99:95,0\t0/1:105:99:55,50
chr1\t15000\trs789012\tC\tT\t3000\tPASS\tDP=250;AF=0.50;AC=3;AN=6\tGT:DP:GQ:AD\t0/1:80:99:40,40\t1/1:85:99:0,85\t0/0:85:99:85,0
chr1\t20000\t.\tATG\tA\t1500\tPASS\tDP=180;AF=0.17;AC=1;AN=6\tGT:DP:GQ:AD\t0/0:60:80:60,0\t0/0:55:75:55,0\t0/1:65:90:40,25
chr1\t25000\trs345678\tG\tGA\t2000\tPASS\tDP=220;AF=0.33;AC=2;AN=6\tGT:DP:GQ:AD\t0/1:75:99:40,35\t0/0:70:99:70,0\t0/1:75:95:45,30
chr2\t50000\trs111222\tT\tC\t800\tLowQual\tDP=50;AF=0.17;AC=1;AN=6\tGT:DP:GQ:AD\t0/0:15:30:15,0\t0/0:20:40:20,0\t0/1:15:20:10,5
chr2\t60000\trs333444\tA\tG,T\t4500\tPASS\tDP=280;AF=0.33,0.17;AC=2,1;AN=6\tGT:DP:GQ:AD\t0/1:90:99:45,45,0\t0/2:95:99:50,0,45\t0/0:95:99:95,0,0
chr3\t100000\t.\tC\tT\t200\tLowQual;LowDP\tDP=20;AF=0.50;AC=1;AN=2\tGT:DP:GQ:AD\t./.:.:.:.\t0/1:20:15:12,8\t./.:.:.:.\t"""

meta, header, variants = parse_vcf(vcf_example)

print(f"Meta-information lines: {len(meta)}")
print(f"Columns: {header}")
print(f"Variants: {len(variants)}\n")

for v in variants:
    alt_str = ','.join(v['alt'])
    print(f"{v['chrom']}:{v['pos']} {v['id']:>12}  {v['ref']}>{alt_str}  "
          f"QUAL={v['qual']}  {v['filter']}  Type={v['type']}  "
          f"AF={v['info'].get('AF', 'N/A')}  DP={v['info'].get('DP', 'N/A')}")

In [ ]:
def decode_genotype(gt_string, ref, alts):
    """Decode a VCF genotype string into alleles."""
    alleles = [ref] + alts
    separator = '|' if '|' in gt_string else '/'
    indices = gt_string.split(separator)
    
    decoded = []
    for idx in indices:
        if idx == '.':
            decoded.append('.')
        else:
            decoded.append(alleles[int(idx)])
    
    phased = separator == '|'
    
    # Determine zygosity
    if '.' in indices:
        zygosity = 'missing'
    elif len(set(indices)) == 1:
        zygosity = 'hom-ref' if indices[0] == '0' else 'hom-alt'
    else:
        zygosity = 'het'
    
    return '/'.join(decoded), zygosity, phased

# Show genotypes for each variant
print("Variant Genotypes Across Samples")
print("=" * 80)

for v in variants:
    print(f"\n{v['chrom']}:{v['pos']} ({v['id']}) {v['ref']}>{','.join(v['alt'])}")
    if 'samples' in v:
        for sample_name, sample_data in v['samples'].items():
            gt = sample_data.get('GT', './.')
            dp = sample_data.get('DP', '.')
            gq = sample_data.get('GQ', '.')
            ad = sample_data.get('AD', '.')
            alleles, zygosity, phased = decode_genotype(gt, v['ref'], v['alt'])
            print(f"  {sample_name}: GT={gt} ({alleles}, {zygosity})  DP={dp}  GQ={gq}  AD={ad}")

---

## 4. Variant Filtering

### 4.1 Why Filter?

Raw variant calls contain false positives from:
- Sequencing errors
- Misalignment (especially in repetitive regions)
- PCR artifacts
- Low-complexity regions

### 4.2 Common Filter Criteria

| Filter | GATK Threshold | Rationale |
|--------|---------------|----------|
| **QUAL** | >= 30 | Overall variant confidence |
| **QD** (QualByDepth) | >= 2.0 | Quality normalized by depth |
| **DP** (Depth) | >= 10 | Minimum read support |
| **FS** (FisherStrand) | <= 60.0 | Strand bias (Fisher's test) |
| **MQ** (MappingQuality) | >= 40.0 | Average mapping quality |
| **MQRankSum** | >= -12.5 | Mapping quality rank sum test |
| **ReadPosRankSum** | >= -8.0 | Read position bias |
| **SOR** (StrandOddsRatio) | <= 3.0 | Strand bias (symmetric odds ratio) |

### 4.3 Genotype-Level Filters

| Filter | Threshold | Rationale |
|--------|----------|----------|
| **GQ** (Genotype Quality) | >= 20 | Confidence in genotype call |
| **DP** (Sample Depth) | >= 10 | Per-sample read depth |
| **AB** (Allele Balance) | 0.2-0.8 for hets | Expected ~0.5 for heterozygous |

In [ ]:
def filter_variants(variants, min_qual=30, min_dp=10, min_gq=20, require_pass=False):
    """
    Apply quality filters to a list of variants.
    
    Returns:
        passed: variants that pass all filters
        failed: dict mapping filter name to failed variants
    """
    passed = []
    failed = defaultdict(list)
    
    for v in variants:
        reasons = []
        
        # QUAL filter
        if v['qual'] is not None and v['qual'] < min_qual:
            reasons.append(f"LowQUAL({v['qual']}<{min_qual})")
        
        # Depth filter (from INFO field)
        dp = int(v['info'].get('DP', 0))
        if dp < min_dp:
            reasons.append(f"LowDP({dp}<{min_dp})")
        
        # FILTER column
        if require_pass and v['filter'] != 'PASS':
            reasons.append(f"NotPASS({v['filter']})")
        
        # Genotype quality filter (check samples)
        if 'samples' in v:
            low_gq_samples = 0
            total_samples = 0
            for sample_data in v['samples'].values():
                gq = sample_data.get('GQ', '.')
                if gq != '.' and gq != '':
                    total_samples += 1
                    if int(gq) < min_gq:
                        low_gq_samples += 1
            if total_samples > 0 and low_gq_samples == total_samples:
                reasons.append(f"AllSamplesLowGQ")
        
        if reasons:
            for r in reasons:
                failed[r].append(v)
        else:
            passed.append(v)
    
    return passed, failed

# Apply filters
passed, failed = filter_variants(variants, min_qual=500, min_dp=30, require_pass=True)

print("Variant Filtering Results")
print("=" * 50)
print(f"Input variants:  {len(variants)}")
print(f"Passed:          {len(passed)}")
print(f"\nFailed filters:")
for reason, vars_list in sorted(failed.items()):
    print(f"  {reason}: {len(vars_list)} variants")

print(f"\nPassed variants:")
for v in passed:
    print(f"  {v['chrom']}:{v['pos']} {v['ref']}>{','.join(v['alt'])} "
          f"QUAL={v['qual']} DP={v['info'].get('DP')}")

In [ ]:
def calculate_allele_balance(ad_string):
    """Calculate allele balance from AD field."""
    counts = [int(x) for x in ad_string.split(',') if x != '.']
    if not counts or sum(counts) == 0:
        return None
    # Allele balance = alt / (ref + alt)
    return counts[1] / sum(counts) if len(counts) > 1 else 0

# Analyze allele balance for heterozygous calls
print("Allele Balance Analysis for Heterozygous Calls")
print("=" * 60)

ab_values = []
for v in variants:
    if 'samples' not in v:
        continue
    for sample_name, sample_data in v['samples'].items():
        gt = sample_data.get('GT', './.')
        ad = sample_data.get('AD', '.')
        
        if '/' not in gt and '|' not in gt:
            continue
        
        sep = '|' if '|' in gt else '/'
        alleles = gt.split(sep)
        
        if alleles[0] != alleles[1] and '.' not in alleles:  # heterozygous
            ab = calculate_allele_balance(ad)
            if ab is not None:
                ab_values.append(ab)
                status = 'OK' if 0.2 <= ab <= 0.8 else 'SUSPECT'
                print(f"  {v['chrom']}:{v['pos']} {sample_name}: GT={gt} AD={ad} "
                      f"AB={ab:.2f} [{status}]")

if ab_values:
    print(f"\nMean allele balance: {np.mean(ab_values):.2f} (expected ~0.50 for hets)")

---

## 5. Variant Annotation

### 5.1 Annotation Databases

| Database | Focus | Key Information |
|----------|-------|-----------------|
| **dbSNP** | All known variants | rsID, allele frequencies, validation |
| **ClinVar** | Clinical significance | Pathogenic/Benign classification, disease associations |
| **gnomAD** | Population frequencies | AF across populations, constraint scores |
| **1000 Genomes** | Population variation | AF in 26 populations |
| **COSMIC** | Cancer variants | Somatic mutations, cancer type associations |
| **OMIM** | Genetic diseases | Gene-disease relationships |

### 5.2 Annotation Tools

| Tool | Source | Features |
|------|--------|----------|
| **VEP** (Ensembl) | Ensembl | Consequence prediction, plugin system, multiple databases |
| **ANNOVAR** | Wang Lab | Functional annotation, database downloads |
| **SnpEff** | Cingolani | Fast, integrated variant effect predictor |
| **SnpSift** | Cingolani | Filter and annotate VCF files |

```bash
# VEP (Ensembl Variant Effect Predictor)
vep -i variants.vcf -o annotated.vcf \
    --cache --assembly GRCh38 \
    --sift b --polyphen b \
    --af --af_gnomad \
    --appris --canonical \
    --vcf

# ANNOVAR
table_annovar.pl variants.vcf humandb/ \
    -buildver hg38 \
    -out annotated \
    -protocol refGene,clinvar_20220320,gnomad30_genome \
    -operation g,f,f \
    -vcfinput
```

### 5.3 Clinical Significance Categories (ClinVar)

| Category | Meaning | Action |
|----------|---------|--------|
| Pathogenic | Disease-causing | Report, clinical follow-up |
| Likely pathogenic | Probably disease-causing | Report with caveat |
| Uncertain significance (VUS) | Not enough evidence | Monitor, do not report as pathogenic |
| Likely benign | Probably harmless | Usually not reported |
| Benign | Harmless | Not reported |

In [ ]:
# Simulated annotation database for demonstration
ANNOTATION_DB = {
    'rs334': {
        'gene': 'HBB', 'consequence': 'missense_variant',
        'hgvs_p': 'p.Glu7Val', 'clinical': 'Pathogenic',
        'disease': 'Sickle cell disease',
        'gnomad_af': 0.0001, 'sift': 'deleterious', 'polyphen': 'probably_damaging'
    },
    'rs1800497': {
        'gene': 'ANKK1', 'consequence': 'missense_variant',
        'hgvs_p': 'p.Glu713Lys', 'clinical': 'Benign',
        'disease': 'Dopamine receptor activity',
        'gnomad_af': 0.24, 'sift': 'tolerated', 'polyphen': 'benign'
    },
    'rs7903146': {
        'gene': 'TCF7L2', 'consequence': 'intron_variant',
        'hgvs_p': None, 'clinical': 'Risk factor',
        'disease': 'Type 2 diabetes',
        'gnomad_af': 0.29, 'sift': None, 'polyphen': None
    },
    'rs1799945': {
        'gene': 'HFE', 'consequence': 'missense_variant',
        'hgvs_p': 'p.His63Asp', 'clinical': 'Risk factor',
        'disease': 'Hereditary hemochromatosis',
        'gnomad_af': 0.14, 'sift': 'tolerated', 'polyphen': 'benign'
    },
    'rs121918506': {
        'gene': 'BRCA2', 'consequence': 'frameshift_variant',
        'hgvs_p': 'p.Ser1982ArgfsTer22', 'clinical': 'Pathogenic',
        'disease': 'Hereditary breast and ovarian cancer',
        'gnomad_af': 0.00002, 'sift': None, 'polyphen': None
    },
    'rs429358': {
        'gene': 'APOE', 'consequence': 'missense_variant',
        'hgvs_p': 'p.Cys130Arg', 'clinical': 'Risk factor',
        'disease': "Alzheimer's disease",
        'gnomad_af': 0.15, 'sift': 'tolerated', 'polyphen': 'benign'
    },
}

def annotate_variant(rsid):
    """Look up variant annotation by rsID."""
    return ANNOTATION_DB.get(rsid)

def format_annotation_report(rsid, annotation):
    """Format a variant annotation report."""
    if annotation is None:
        return f"{rsid}: Not found in database\n"
    
    lines = [
        f"Variant: {rsid}",
        f"  Gene:          {annotation['gene']}",
        f"  Consequence:   {annotation['consequence']}",
        f"  Protein:       {annotation['hgvs_p'] or 'N/A'}",
        f"  Clinical:      {annotation['clinical']}",
        f"  Disease:       {annotation['disease']}",
        f"  gnomAD AF:     {annotation['gnomad_af']:.6f} ({annotation['gnomad_af']*100:.4f}%)",
        f"  SIFT:          {annotation['sift'] or 'N/A'}",
        f"  PolyPhen:      {annotation['polyphen'] or 'N/A'}",
    ]
    return '\n'.join(lines)

# Look up all variants
print("Variant Annotation Reports")
print("=" * 60)
for rsid in ANNOTATION_DB:
    ann = annotate_variant(rsid)
    print(format_annotation_report(rsid, ann))
    print()

In [ ]:
# Variant effect prediction on coding sequences

CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

AA_NAMES = {
    'A': 'Ala', 'R': 'Arg', 'N': 'Asn', 'D': 'Asp', 'C': 'Cys',
    'E': 'Glu', 'Q': 'Gln', 'G': 'Gly', 'H': 'His', 'I': 'Ile',
    'L': 'Leu', 'K': 'Lys', 'M': 'Met', 'F': 'Phe', 'P': 'Pro',
    'S': 'Ser', 'T': 'Thr', 'W': 'Trp', 'Y': 'Tyr', 'V': 'Val',
    '*': 'Stop'
}

def predict_coding_effect(ref_codon, position_in_codon, alt_base):
    """
    Predict the effect of a SNP on a coding sequence.
    
    Parameters:
        ref_codon: 3-letter codon (e.g., 'ATG')
        position_in_codon: 0, 1, or 2
        alt_base: alternative base (e.g., 'G')
    """
    ref_codon = ref_codon.upper()
    alt_base = alt_base.upper()
    
    alt_codon = list(ref_codon)
    alt_codon[position_in_codon] = alt_base
    alt_codon = ''.join(alt_codon)
    
    ref_aa = CODON_TABLE[ref_codon]
    alt_aa = CODON_TABLE[alt_codon]
    
    if ref_aa == alt_aa:
        effect = 'synonymous'
    elif alt_aa == '*':
        effect = 'nonsense (stop gained)'
    elif ref_aa == '*':
        effect = 'stop lost'
    else:
        effect = 'missense'
    
    return {
        'ref_codon': ref_codon,
        'alt_codon': alt_codon,
        'ref_aa': f"{AA_NAMES[ref_aa]} ({ref_aa})",
        'alt_aa': f"{AA_NAMES[alt_aa]} ({alt_aa})",
        'effect': effect,
    }

# Demonstrate with classic examples
print("Coding Variant Effect Predictions")
print("=" * 60)

examples = [
    ('GAG', 0, 'A', 'Sickle cell (HBB E6V)'),
    ('GCA', 2, 'G', 'Synonymous (Ala->Ala)'),
    ('TGG', 2, 'A', 'Nonsense (Trp->Stop)'),
    ('TAA', 0, 'C', 'Stop lost (Stop->Gln)'),
    ('CGT', 1, 'A', 'Missense (Arg->His)'),
]

for codon, pos, alt, label in examples:
    result = predict_coding_effect(codon, pos, alt)
    print(f"\n{label}:")
    print(f"  {result['ref_codon']} -> {result['alt_codon']}")
    print(f"  {result['ref_aa']} -> {result['alt_aa']}")
    print(f"  Effect: {result['effect']}")

---

## 6. Population Genetics

### 6.1 Allele Frequency

**Allele frequency** is the proportion of a specific allele among all alleles in a population.

```
For a biallelic SNP with alleles A and G:

  Individuals:  AA  AG  AG  GG  AA  AG  AA  AG  AA  GG
  
  Count alleles:
  A alleles = 2+1+1+0+2+1+2+1+2+0 = 12
  G alleles = 0+1+1+2+0+1+0+1+0+2 = 8
  Total = 20
  
  p(A) = 12/20 = 0.60
  q(G) = 8/20 = 0.40
  p + q = 1.0
```

### 6.2 Hardy-Weinberg Equilibrium (HWE)

Under HWE, genotype frequencies are predictable from allele frequencies:

```
If p = freq(A), q = freq(G), p + q = 1:

  freq(AA) = p^2
  freq(AG) = 2pq
  freq(GG) = q^2
  
  p^2 + 2pq + q^2 = 1
```

**HWE assumptions** (deviation suggests evolutionary forces at work):
1. No mutation
2. No natural selection
3. Random mating
4. Large population (no genetic drift)
5. No migration (no gene flow)

**In variant calling:** HWE deviations may indicate genotyping errors or population structure.

In [ ]:
from scipy.stats import chi2

def calculate_allele_frequencies(genotypes):
    """
    Calculate allele frequencies from a list of genotypes.
    Genotypes: list of tuples, e.g., [(0,0), (0,1), (1,1)]
    where 0 = reference allele, 1 = alternate allele.
    """
    total_alleles = 0
    alt_count = 0
    
    for g in genotypes:
        if None in g:  # skip missing
            continue
        total_alleles += 2
        alt_count += sum(g)
    
    q = alt_count / total_alleles  # alt allele frequency
    p = 1 - q  # ref allele frequency
    return p, q

def hardy_weinberg_test(genotypes):
    """
    Test for Hardy-Weinberg Equilibrium using chi-squared test.
    
    Returns: dict with observed/expected counts, chi-squared statistic, p-value
    """
    # Count genotypes
    valid = [g for g in genotypes if None not in g]
    n = len(valid)
    
    obs_hom_ref = sum(1 for g in valid if g == (0, 0))
    obs_het = sum(1 for g in valid if g == (0, 1) or g == (1, 0))
    obs_hom_alt = sum(1 for g in valid if g == (1, 1))
    
    # Allele frequencies
    p, q = calculate_allele_frequencies(valid)
    
    # Expected under HWE
    exp_hom_ref = p**2 * n
    exp_het = 2 * p * q * n
    exp_hom_alt = q**2 * n
    
    # Chi-squared test (1 degree of freedom for HWE)
    observed = [obs_hom_ref, obs_het, obs_hom_alt]
    expected = [exp_hom_ref, exp_het, exp_hom_alt]
    
    chi2_stat = sum((o - e)**2 / e for o, e in zip(observed, expected) if e > 0)
    p_value = 1 - chi2.cdf(chi2_stat, df=1)
    
    return {
        'n': n,
        'p': p, 'q': q,
        'observed': {'hom_ref': obs_hom_ref, 'het': obs_het, 'hom_alt': obs_hom_alt},
        'expected': {'hom_ref': exp_hom_ref, 'het': exp_het, 'hom_alt': exp_hom_alt},
        'chi2': chi2_stat,
        'p_value': p_value,
        'in_hwe': p_value > 0.05
    }

# Example 1: Population in HWE
np.random.seed(42)
p_true = 0.7
q_true = 0.3
n_individuals = 500

# Simulate genotypes under HWE
hwe_genotypes = []
for _ in range(n_individuals):
    allele1 = 0 if random.random() < p_true else 1
    allele2 = 0 if random.random() < p_true else 1
    hwe_genotypes.append((allele1, allele2))

result1 = hardy_weinberg_test(hwe_genotypes)

print("Example 1: Population simulated under HWE")
print("=" * 55)
print(f"N = {result1['n']}, p(ref) = {result1['p']:.3f}, q(alt) = {result1['q']:.3f}")
print(f"\n{'':>12} {'Obs':>8} {'Exp (HWE)':>12}")
print(f"{'Hom Ref':>12} {result1['observed']['hom_ref']:>8} {result1['expected']['hom_ref']:>12.1f}")
print(f"{'Het':>12} {result1['observed']['het']:>8} {result1['expected']['het']:>12.1f}")
print(f"{'Hom Alt':>12} {result1['observed']['hom_alt']:>8} {result1['expected']['hom_alt']:>12.1f}")
print(f"\nChi-squared = {result1['chi2']:.3f}, p-value = {result1['p_value']:.4f}")
print(f"Conclusion: {'Consistent with HWE' if result1['in_hwe'] else 'Deviates from HWE'}")

In [ ]:
# Example 2: Population with excess heterozygosity (e.g., balancing selection)
excess_het_genotypes = (
    [(0, 0)] * 100 +  # fewer hom_ref than expected
    [(0, 1)] * 300 +  # more hets than expected
    [(1, 1)] * 100    # fewer hom_alt than expected
)

result2 = hardy_weinberg_test(excess_het_genotypes)

print("\nExample 2: Population with excess heterozygosity")
print("=" * 55)
print(f"N = {result2['n']}, p(ref) = {result2['p']:.3f}, q(alt) = {result2['q']:.3f}")
print(f"\n{'':>12} {'Obs':>8} {'Exp (HWE)':>12}")
print(f"{'Hom Ref':>12} {result2['observed']['hom_ref']:>8} {result2['expected']['hom_ref']:>12.1f}")
print(f"{'Het':>12} {result2['observed']['het']:>8} {result2['expected']['het']:>12.1f}")
print(f"{'Hom Alt':>12} {result2['observed']['hom_alt']:>8} {result2['expected']['hom_alt']:>12.1f}")
print(f"\nChi-squared = {result2['chi2']:.3f}, p-value = {result2['p_value']:.6f}")
print(f"Conclusion: {'Consistent with HWE' if result2['in_hwe'] else 'DEVIATES from HWE (excess heterozygosity)'}")

In [ ]:
# Visualize HWE
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: HWE curve
p_range = np.linspace(0, 1, 100)
q_range = 1 - p_range

axes[0].plot(p_range, p_range**2, 'b-', linewidth=2, label='Hom Ref (p$^2$)')
axes[0].plot(p_range, 2 * p_range * q_range, 'g-', linewidth=2, label='Het (2pq)')
axes[0].plot(p_range, q_range**2, 'r-', linewidth=2, label='Hom Alt (q$^2$)')
axes[0].set_xlabel('Reference Allele Frequency (p)')
axes[0].set_ylabel('Genotype Frequency')
axes[0].set_title('Hardy-Weinberg Equilibrium')
axes[0].legend()
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Plot 2: Observed vs expected for our examples
x = np.arange(3)
width = 0.35
labels = ['Hom Ref', 'Het', 'Hom Alt']

obs_vals = [result1['observed']['hom_ref'], result1['observed']['het'], result1['observed']['hom_alt']]
exp_vals = [result1['expected']['hom_ref'], result1['expected']['het'], result1['expected']['hom_alt']]

axes[1].bar(x - width/2, obs_vals, width, label='Observed', color='steelblue', edgecolor='black')
axes[1].bar(x + width/2, exp_vals, width, label='Expected (HWE)', color='coral', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Count')
axes[1].set_title(f'HWE Test (p-value = {result1["p_value"]:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

### 6.3 Population Allele Frequency Databases

**1000 Genomes Project** populations:

| Super Population | Code | Populations |
|-----------------|------|-------------|
| African | AFR | YRI, LWK, GWD, MSL, ESN, ASW, ACB |
| American | AMR | MXL, PUR, CLM, PEL |
| East Asian | EAS | CHB, JPT, CHS, CDX, KHV |
| European | EUR | CEU, TSI, FIN, GBR, IBS |
| South Asian | SAS | GIH, PJL, BEB, STU, ITU |

Allele frequencies can vary dramatically between populations, which is critical for:
- Interpreting clinical variants (common in one population, rare in another)
- GWAS study design
- Forensic genetics

In [ ]:
# Simulated population-specific allele frequencies
population_frequencies = {
    'rs334 (HBB - Sickle Cell)': {
        'AFR': 0.10, 'AMR': 0.03, 'EAS': 0.001, 'EUR': 0.001, 'SAS': 0.005
    },
    'rs1800497 (ANKK1 - DRD2)': {
        'AFR': 0.18, 'AMR': 0.30, 'EAS': 0.35, 'EUR': 0.20, 'SAS': 0.32
    },
    'rs7903146 (TCF7L2 - T2D)': {
        'AFR': 0.30, 'AMR': 0.25, 'EAS': 0.03, 'EUR': 0.29, 'SAS': 0.31
    },
    'rs429358 (APOE e4)': {
        'AFR': 0.27, 'AMR': 0.10, 'EAS': 0.09, 'EUR': 0.15, 'SAS': 0.08
    },
}

# Plot population frequencies
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

pops = ['AFR', 'AMR', 'EAS', 'EUR', 'SAS']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

for idx, (variant, freqs) in enumerate(population_frequencies.items()):
    ax = axes[idx]
    values = [freqs[p] for p in pops]
    bars = ax.bar(pops, values, color=colors, edgecolor='black', alpha=0.8)
    ax.set_ylabel('Allele Frequency')
    ax.set_title(variant)
    ax.set_ylim(0, max(values) * 1.3)
    
    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Allele Frequencies Across Populations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 7. GWAS Concepts

### 7.1 Genome-Wide Association Studies

GWAS tests millions of SNPs for association with a trait or disease.

```
Study Design:
  Cases (affected):    N = 10,000+
  Controls (healthy):  N = 10,000+
  
  Genotype millions of SNPs in all subjects
  
  For each SNP, test: Is allele frequency different between cases and controls?
  
  Statistical test: Chi-squared test or logistic regression
  
  Multiple testing correction: ~1 million independent tests
  Genome-wide significance threshold: p < 5 x 10^-8
```

### 7.2 Manhattan Plot

The Manhattan plot shows -log10(p-value) for each SNP, ordered by chromosomal position. Peaks above the significance threshold indicate associated loci.

### 7.3 QQ Plot

The QQ (quantile-quantile) plot compares observed p-values against expected p-values under the null hypothesis. It helps detect:
- **Genomic inflation**: Systematic deviation suggests population stratification or technical artifacts
- **True signal**: Deviation at the tail (extreme values) is expected if there are true associations

In [ ]:
def simulate_gwas(n_snps=100000, n_causal=10, effect_size=0.5):
    """
    Simulate GWAS results with a few causal variants.
    
    Returns: DataFrame-like list of dicts with chr, pos, p-value
    """
    results = []
    snps_per_chr = n_snps // 22
    
    # Select causal SNPs
    causal_indices = set(random.sample(range(n_snps), n_causal))
    
    snp_idx = 0
    for chrom in range(1, 23):
        chr_size = random.randint(50_000_000, 250_000_000)
        for j in range(snps_per_chr):
            pos = int((j + 1) / snps_per_chr * chr_size)
            
            if snp_idx in causal_indices:
                # Causal SNP: very low p-value
                p_val = 10 ** (-random.uniform(8, 20))
            else:
                # Null SNP: uniform p-value (with some noise)
                p_val = random.random()
            
            results.append({
                'chr': chrom,
                'pos': pos,
                'p_value': p_val,
                'is_causal': snp_idx in causal_indices
            })
            snp_idx += 1
    
    return results

gwas_results = simulate_gwas(n_snps=50000, n_causal=8)
print(f"Simulated {len(gwas_results)} SNPs across 22 chromosomes")
print(f"Causal SNPs: {sum(1 for r in gwas_results if r['is_causal'])}")

sig = [r for r in gwas_results if r['p_value'] < 5e-8]
print(f"Genome-wide significant (p < 5e-8): {len(sig)}")
for s in sig:
    print(f"  chr{s['chr']}:{s['pos']} p={s['p_value']:.2e} {'(CAUSAL)' if s['is_causal'] else ''}")

In [ ]:
def manhattan_plot(results, significance_threshold=5e-8, suggestive_threshold=1e-5):
    """Create a Manhattan plot from GWAS results."""
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Calculate cumulative positions for x-axis
    chr_offsets = {}
    cumulative_pos = 0
    
    # Get max position per chromosome
    chr_max = defaultdict(int)
    for r in results:
        chr_max[r['chr']] = max(chr_max[r['chr']], r['pos'])
    
    for chrom in sorted(chr_max.keys()):
        chr_offsets[chrom] = cumulative_pos
        cumulative_pos += chr_max[chrom]
    
    # Plot points
    colors = ['#1f77b4', '#aec7e8']  # alternating colors
    x_positions = []
    y_values = []
    point_colors = []
    
    for r in results:
        x = chr_offsets[r['chr']] + r['pos']
        y = -np.log10(max(r['p_value'], 1e-300))  # avoid log(0)
        x_positions.append(x)
        y_values.append(y)
        point_colors.append(colors[r['chr'] % 2])
    
    ax.scatter(x_positions, y_values, c=point_colors, s=3, alpha=0.5)
    
    # Highlight significant SNPs
    for r in results:
        if r['p_value'] < significance_threshold:
            x = chr_offsets[r['chr']] + r['pos']
            y = -np.log10(r['p_value'])
            ax.scatter(x, y, c='red', s=40, zorder=5, edgecolors='black', linewidth=0.5)
    
    # Threshold lines
    ax.axhline(y=-np.log10(significance_threshold), color='red', linestyle='--',
               linewidth=1, label=f'Genome-wide (p={significance_threshold:.0e})')
    ax.axhline(y=-np.log10(suggestive_threshold), color='blue', linestyle=':',
               linewidth=1, label=f'Suggestive (p={suggestive_threshold:.0e})')
    
    # Chromosome labels
    chr_centers = []
    for chrom in sorted(chr_max.keys()):
        center = chr_offsets[chrom] + chr_max[chrom] / 2
        chr_centers.append(center)
    ax.set_xticks(chr_centers)
    ax.set_xticklabels([str(c) for c in sorted(chr_max.keys())], fontsize=8)
    
    ax.set_xlabel('Chromosome')
    ax.set_ylabel('-log$_{10}$(p-value)')
    ax.set_title('Manhattan Plot - GWAS Results')
    ax.legend(loc='upper right')
    ax.set_ylim(0, max(y_values) * 1.1)
    
    plt.tight_layout()
    plt.show()

manhattan_plot(gwas_results)

In [ ]:
def qq_plot(p_values):
    """Create a QQ plot for GWAS p-values."""
    # Sort p-values
    observed = sorted(p_values)
    n = len(observed)
    
    # Expected p-values under null
    expected = [(i + 0.5) / n for i in range(n)]
    
    # Convert to -log10
    obs_log = [-np.log10(max(p, 1e-300)) for p in observed]
    exp_log = [-np.log10(p) for p in expected]
    
    # Calculate genomic inflation factor (lambda)
    from scipy.stats import chi2 as chi2_dist
    chi2_values = [chi2_dist.ppf(1 - p, 1) for p in observed if p > 0]
    lambda_gc = np.median(chi2_values) / chi2_dist.ppf(0.5, 1)
    
    fig, ax = plt.subplots(figsize=(7, 7))
    
    ax.scatter(exp_log, obs_log, s=4, alpha=0.5, color='steelblue')
    
    # Identity line
    max_val = max(max(exp_log), max(obs_log))
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Expected (null)')
    
    # Significance threshold
    ax.axhline(y=-np.log10(5e-8), color='gray', linestyle=':', alpha=0.5)
    
    ax.set_xlabel('Expected -log$_{10}$(p-value)')
    ax.set_ylabel('Observed -log$_{10}$(p-value)')
    ax.set_title(f'QQ Plot (genomic inflation $\\lambda$ = {lambda_gc:.3f})')
    ax.legend()
    ax.set_aspect('equal')
    ax.set_xlim(0, max(exp_log) * 1.05)
    ax.set_ylim(0, max(obs_log) * 1.05)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Genomic inflation factor (lambda_GC): {lambda_gc:.3f}")
    if lambda_gc < 1.05:
        print("  -> Good: No evidence of systematic inflation")
    elif lambda_gc < 1.10:
        print("  -> Mild inflation: Consider population stratification correction")
    else:
        print("  -> Significant inflation: Population stratification or batch effects likely")

p_values = [r['p_value'] for r in gwas_results]
qq_plot(p_values)

### 7.4 GWAS Interpretation

**Key concepts:**

| Term | Definition |
|------|------------|
| **Lead SNP** | Most significant SNP at a locus |
| **Linkage Disequilibrium (LD)** | Non-random association between nearby variants |
| **Odds Ratio (OR)** | Effect size: OR > 1 = increased risk, OR < 1 = decreased risk |
| **Genomic inflation (lambda)** | Measure of systematic p-value inflation |
| **Fine mapping** | Identifying the causal variant among correlated SNPs |
| **Polygenic Risk Score (PRS)** | Combined effect of many variants on disease risk |

**Limitations of GWAS:**
- Association is not causation
- Most GWAS hits are in non-coding regions
- Common variants typically have small effect sizes
- Population-specific results may not generalize
- Missing heritability: GWAS explains only a fraction of trait heritability

---

## 8. Practical: Building a Variant Analysis Pipeline

Let us build a complete Python-based variant analysis workflow.

In [ ]:
def generate_simulated_vcf(n_variants=200, n_samples=10):
    """Generate a realistic simulated VCF dataset."""
    chromosomes = [f'chr{i}' for i in range(1, 23)]
    
    records = []
    for i in range(n_variants):
        chrom = random.choice(chromosomes)
        pos = random.randint(10000, 200_000_000)
        ref = random.choice('ACGT')
        
        # Variant type
        vtype = random.choices(['snv', 'ins', 'del'], weights=[80, 10, 10])[0]
        if vtype == 'snv':
            alt = random.choice([b for b in 'ACGT' if b != ref])
        elif vtype == 'ins':
            insert_len = random.randint(1, 5)
            alt = ref + ''.join(random.choice('ACGT') for _ in range(insert_len))
        else:
            del_len = random.randint(1, 5)
            ref = ref + ''.join(random.choice('ACGT') for _ in range(del_len))
            alt = ref[0]
        
        # Allele frequency (most variants are rare)
        af = random.choices(
            [random.uniform(0.001, 0.01), random.uniform(0.01, 0.05),
             random.uniform(0.05, 0.50)],
            weights=[50, 30, 20]
        )[0]
        
        # Quality and depth
        dp = random.randint(5, 300)
        qual = random.uniform(10, 10000)
        
        # Generate genotypes for each sample
        genotypes = []
        for s in range(n_samples):
            r = random.random()
            if r < (1 - af)**2:
                gt = (0, 0)
            elif r < (1 - af)**2 + 2 * af * (1 - af):
                gt = (0, 1)
            else:
                gt = (1, 1)
            genotypes.append(gt)
        
        # Assign rsID to some variants
        rsid = f'rs{random.randint(1000, 99999999)}' if random.random() < 0.6 else '.'
        
        # Random clinical significance for annotated variants
        clinical = random.choices(
            ['Benign', 'Likely_benign', 'VUS', 'Likely_pathogenic', 'Pathogenic', None],
            weights=[30, 20, 20, 5, 2, 23]
        )[0]
        
        records.append({
            'chrom': chrom, 'pos': pos, 'id': rsid,
            'ref': ref, 'alt': alt,
            'qual': qual, 'dp': dp, 'af': af,
            'genotypes': genotypes,
            'clinical': clinical,
            'type': classify_variant(ref, alt)
        })
    
    # Sort by chromosome and position
    chrom_order = {f'chr{i}': i for i in range(1, 23)}
    records.sort(key=lambda r: (chrom_order.get(r['chrom'], 99), r['pos']))
    
    return records

sim_variants = generate_simulated_vcf(n_variants=500, n_samples=10)

print(f"Generated {len(sim_variants)} simulated variants")
print(f"\nVariant type distribution:")
type_counts = Counter(v['type'] for v in sim_variants)
for vtype, count in type_counts.most_common():
    print(f"  {vtype}: {count}")

print(f"\nWith rsID: {sum(1 for v in sim_variants if v['id'] != '.')}")
print(f"With clinical annotation: {sum(1 for v in sim_variants if v['clinical'])}")

In [ ]:
# Comprehensive variant analysis

def analyze_variants(variants):
    """Perform comprehensive analysis of a variant dataset."""
    print("=" * 60)
    print("       VARIANT ANALYSIS REPORT")
    print("=" * 60)
    
    # 1. Summary statistics
    print(f"\n1. SUMMARY")
    print(f"   Total variants: {len(variants)}")
    print(f"   Chromosomes: {len(set(v['chrom'] for v in variants))}")
    
    # Type distribution
    print(f"\n2. VARIANT TYPES")
    for vtype, count in Counter(v['type'] for v in variants).most_common():
        print(f"   {vtype:>12}: {count:>5} ({100*count/len(variants):.1f}%)")
    
    # Quality distribution
    quals = [v['qual'] for v in variants]
    print(f"\n3. QUALITY")
    print(f"   Mean QUAL: {np.mean(quals):.1f}")
    print(f"   Median QUAL: {np.median(quals):.1f}")
    print(f"   QUAL >= 30: {sum(1 for q in quals if q >= 30)} ({100*sum(1 for q in quals if q >= 30)/len(quals):.1f}%)")
    
    # Depth distribution
    depths = [v['dp'] for v in variants]
    print(f"\n4. DEPTH")
    print(f"   Mean DP: {np.mean(depths):.1f}")
    print(f"   DP >= 10: {sum(1 for d in depths if d >= 10)} ({100*sum(1 for d in depths if d >= 10)/len(depths):.1f}%)")
    print(f"   DP >= 30: {sum(1 for d in depths if d >= 30)} ({100*sum(1 for d in depths if d >= 30)/len(depths):.1f}%)")
    
    # Allele frequency distribution
    afs = [v['af'] for v in variants]
    print(f"\n5. ALLELE FREQUENCY")
    print(f"   Rare (AF < 0.01): {sum(1 for af in afs if af < 0.01)}")
    print(f"   Low-freq (0.01-0.05): {sum(1 for af in afs if 0.01 <= af < 0.05)}")
    print(f"   Common (AF >= 0.05): {sum(1 for af in afs if af >= 0.05)}")
    
    # Clinical significance
    clinical = Counter(v['clinical'] for v in variants if v['clinical'])
    print(f"\n6. CLINICAL SIGNIFICANCE")
    for cat, count in clinical.most_common():
        print(f"   {cat:>20}: {count}")
    
    # Ts/Tv ratio (for SNVs)
    transitions = {'AG', 'GA', 'CT', 'TC'}
    ts = 0
    tv = 0
    for v in variants:
        if v['type'] == 'SNV':
            change = v['ref'] + v['alt']
            if change in transitions:
                ts += 1
            else:
                tv += 1
    tstv = ts / tv if tv > 0 else 0
    print(f"\n7. TRANSITION/TRANSVERSION RATIO")
    print(f"   Transitions: {ts}")
    print(f"   Transversions: {tv}")
    print(f"   Ts/Tv ratio: {tstv:.2f} (expected ~2.0-2.1 for WGS, ~2.8-3.0 for WES)")
    
    print("\n" + "=" * 60)

analyze_variants(sim_variants)

In [ ]:
# Visualization dashboard
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Variant type pie chart
type_counts = Counter(v['type'] for v in sim_variants)
axes[0, 0].pie(type_counts.values(), labels=type_counts.keys(), autopct='%1.1f%%',
               colors=['steelblue', 'coral', 'seagreen', 'gold'])
axes[0, 0].set_title('Variant Types')

# 2. Quality distribution
axes[0, 1].hist([v['qual'] for v in sim_variants], bins=50, edgecolor='black',
               alpha=0.7, color='steelblue')
axes[0, 1].axvline(x=30, color='red', linestyle='--', label='Q30')
axes[0, 1].set_xlabel('QUAL Score')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Quality Distribution')
axes[0, 1].legend()

# 3. Depth distribution
axes[0, 2].hist([v['dp'] for v in sim_variants], bins=50, edgecolor='black',
               alpha=0.7, color='seagreen')
axes[0, 2].axvline(x=10, color='red', linestyle='--', label='DP=10')
axes[0, 2].axvline(x=30, color='orange', linestyle='--', label='DP=30')
axes[0, 2].set_xlabel('Read Depth (DP)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('Depth Distribution')
axes[0, 2].legend()

# 4. Allele frequency distribution (log scale)
afs = [v['af'] for v in sim_variants]
axes[1, 0].hist(afs, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1, 0].set_xlabel('Allele Frequency')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Allele Frequency Spectrum')

# 5. Variants per chromosome
chr_counts = Counter(v['chrom'] for v in sim_variants)
chr_labels = [f'{i}' for i in range(1, 23)]
chr_vals = [chr_counts.get(f'chr{i}', 0) for i in range(1, 23)]
axes[1, 1].bar(chr_labels, chr_vals, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 1].set_xlabel('Chromosome')
axes[1, 1].set_ylabel('Variant Count')
axes[1, 1].set_title('Variants per Chromosome')
axes[1, 1].tick_params(axis='x', rotation=45)

# 6. Clinical significance
clinical_counts = Counter(v['clinical'] for v in sim_variants if v['clinical'])
colors_clinical = {
    'Pathogenic': '#d62728', 'Likely_pathogenic': '#ff7f0e',
    'VUS': '#bcbd22', 'Likely_benign': '#2ca02c', 'Benign': '#1f77b4'
}
c_labels = list(clinical_counts.keys())
c_values = list(clinical_counts.values())
c_colors = [colors_clinical.get(l, 'gray') for l in c_labels]
axes[1, 2].barh(c_labels, c_values, color=c_colors, edgecolor='black', alpha=0.8)
axes[1, 2].set_xlabel('Count')
axes[1, 2].set_title('Clinical Significance')

plt.suptitle('Variant Analysis Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Exercises

### Exercise 1: VCF Parser

Write a function `parse_vcf_file(filepath)` that reads a VCF file and returns:
- Total number of variants
- Number of SNVs, insertions, deletions
- Number of PASS variants
- Average quality score
- Number of novel variants (ID = '.')

Test it on the VCF example text from this notebook.

In [ ]:
# Exercise 1: Your solution here

def vcf_summary(vcf_text):
    """Parse VCF and return summary statistics."""
    # YOUR CODE HERE
    pass

# Test with vcf_example from above
# vcf_summary(vcf_example)

### Exercise 2: Variant Quality Filter

Implement a configurable variant filter that accepts:
- Minimum QUAL score
- Minimum depth (DP)
- Minimum allele frequency
- Maximum allele frequency (to exclude common variants)
- Required FILTER status

Apply it to `sim_variants` with clinical-grade filtering (QUAL>=100, DP>=20, AF<=0.01) and report how many rare, high-quality variants remain.

In [ ]:
# Exercise 2: Your solution here

def clinical_variant_filter(variants, min_qual=100, min_dp=20,
                            min_af=None, max_af=0.01):
    """Filter variants for clinical analysis."""
    # YOUR CODE HERE
    pass

# Test
# filtered = clinical_variant_filter(sim_variants)
# print(f"Filtered: {len(filtered)} of {len(sim_variants)} variants")

### Exercise 3: Transition/Transversion Analysis

Write a function that:
1. Calculates the Ts/Tv ratio for a set of SNVs
2. Breaks down all 12 possible substitution types (A>C, A>G, A>T, ...)
3. Creates a bar plot of substitution types
4. Determines if the Ts/Tv ratio is in the expected range (2.0-2.1 for WGS, 2.8-3.0 for WES)

Apply it to the SNVs in `sim_variants`.

In [ ]:
# Exercise 3: Your solution here

def tstv_analysis(variants):
    """Analyze transition/transversion ratio and substitution spectrum."""
    # YOUR CODE HERE
    pass

# Test
# tstv_analysis(sim_variants)

### Exercise 4: Hardy-Weinberg Filter

Write a function that:
1. Takes a list of genotypes for a variant
2. Performs a Hardy-Weinberg equilibrium test
3. Returns whether the variant passes the HWE filter (p > 0.001)

Apply it as an additional quality filter: variants that deviate from HWE may indicate genotyping errors.

Test with these genotype sets:
- Set A: 40 AA, 45 AG, 15 GG (should pass HWE)
- Set B: 80 AA, 5 AG, 15 GG (should fail HWE -- deficit of heterozygotes)

In [ ]:
# Exercise 4: Your solution here

def hwe_filter(genotypes, threshold=0.001):
    """Test if genotypes are consistent with Hardy-Weinberg Equilibrium."""
    # YOUR CODE HERE
    pass

# Test
# set_a = [(0,0)]*40 + [(0,1)]*45 + [(1,1)]*15
# set_b = [(0,0)]*80 + [(0,1)]*5 + [(1,1)]*15
# print(f"Set A (balanced): passes HWE = {hwe_filter(set_a)}")
# print(f"Set B (het deficit): passes HWE = {hwe_filter(set_b)}")

### Exercise 5: Variant Annotation Pipeline

Build a mini annotation pipeline that:
1. Takes a list of variant records
2. Classifies each variant (SNV/indel/etc.)
3. For SNVs in coding regions, predicts the protein effect (missense/synonymous/nonsense)
4. Assigns a severity score: Pathogenic > Likely_pathogenic > VUS > Likely_benign > Benign
5. Returns a ranked list of the most clinically interesting variants

Use the `sim_variants` dataset and find the top 10 most concerning variants (pathogenic/likely pathogenic with low allele frequency).

In [ ]:
# Exercise 5: Your solution here

def annotate_and_rank(variants):
    """Annotate variants and rank by clinical importance."""
    # YOUR CODE HERE
    pass

# Test
# top_variants = annotate_and_rank(sim_variants)
# for v in top_variants[:10]:
#     print(v)

### Exercise 6: Mini-GWAS Simulation

Simulate a case-control GWAS:
1. Generate 100 cases and 100 controls
2. For 1000 SNPs, simulate genotypes under HWE
3. For 3 "causal" SNPs, increase the alt allele frequency in cases (odds ratio = 2.0)
4. Perform a chi-squared test for each SNP
5. Create a Manhattan-style plot
6. Determine if your causal SNPs are detected at p < 0.05 / 1000 (Bonferroni)

In [ ]:
# Exercise 6: Your solution here

def mini_gwas(n_cases=100, n_controls=100, n_snps=1000, n_causal=3):
    """Simulate and run a mini GWAS."""
    # YOUR CODE HERE
    pass

# mini_gwas()

---

## Resources

### Tools
- [GATK](https://gatk.broadinstitute.org/) - Genome Analysis Toolkit
- [bcftools](http://www.htslib.org/) - VCF/BCF manipulation
- [FreeBayes](https://github.com/freebayes/freebayes) - Bayesian variant caller
- [DeepVariant](https://github.com/google/deepvariant) - Deep learning variant caller
- [VEP](https://www.ensembl.org/vep) - Variant Effect Predictor
- [ANNOVAR](https://annovar.openbioinformatics.org/) - Functional annotation
- [SnpEff](http://pcingola.github.io/SnpEff/) - Variant annotation and effect prediction

### Databases
- [dbSNP](https://www.ncbi.nlm.nih.gov/snp/) - Variant database
- [ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) - Clinical variant interpretations
- [gnomAD](https://gnomad.broadinstitute.org/) - Population allele frequencies
- [GWAS Catalog](https://www.ebi.ac.uk/gwas/) - Published GWAS results
- [COSMIC](https://cancer.sanger.ac.uk/cosmic) - Cancer mutations
- [OMIM](https://www.omim.org/) - Genetic disorders

### Specifications
- [VCF Format Specification (v4.2)](https://samtools.github.io/hts-specs/VCFv4.2.pdf)
- [VCF Format Specification (v4.3)](https://samtools.github.io/hts-specs/VCFv4.3.pdf)

### Tutorials
- [GATK Best Practices](https://gatk.broadinstitute.org/hc/en-us/sections/360007226651-Best-Practices-Workflows)
- [GWAS Tutorial (Marees et al. 2018)](https://doi.org/10.1002/mpr.1608)
- [Population Genetics in Bioinformatics](https://www.internationalgenome.org/)

---

[< Previous: Bioinformatics Data Formats: A Comprehensive Guide](../01_NGS_Fundamentals/02_bio_data_formats.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: SNP Calling Pipeline: A Practical Hands-On Work... >](02_snp_calling_pipeline.ipynb)

---